In [1]:
from sympy import *
from random import randint
%run Geom_Prolongation.ipynb
%run Particular_Distributions.ipynb
%run CartanGeometry.ipynb

In [2]:
g=Symp_symb(7)
E=g.ext_alg
C=g.cochain_complex
D,D_JS=Standard_Prenorm_distr(g,9,unshelve=True)
P=Geom_Prolongation(D,D_JS)
G=RegularCartanGeometry(g,'eta')
D_G=Distr_of_constant_symbol(g,-G.curvature)
y,h,e=symbols('y,h,e')
K=IndexedBase('K')
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis

In [3]:
# unshelve
with shelve.open('harm_invars') as shelf:
    exec("F9=shelf['Normal_frame_9']")
    exec("K9=shelf['Normal_curv_9']")

## Checking Bianchi for normal frame via abstract Bianchi

In [4]:
tuples_by_wght={}
for i in range(3,len(g.basis)):
    for j in range(i+1,len(g.basis)):
        for k in range(j+1,len(g.basis)):
            for m in range(len(g.basis)):
                w=-g.basis[i].wght-g.basis[j].wght-g.basis[k].wght+g.basis[m].wght
                if w not in tuples_by_wght: tuples_by_wght[w]=[]
                tuples_by_wght[w].append((i,j,k,m))

In [5]:
def der_term(G,i0,i1,i2):
    """Returns the derivative term for the component of the Bianchi identity
    applied to (i0,i1,i2), which is an element of G.symbol."""
    g=G.symbol
    X0,X1,X2=[g.basis[a] for a in [i0,i1,i2]]
    w0,w1,w2=[None,None,None]
    try: w0=g.ext_alg.elt_from_cd({(str(X1),str(X2)):1})
    except(KeyError): pass
    try: w1=g.ext_alg.elt_from_cd({(str(X2),str(X0)):1})
    except(KeyError): pass
    try: w2=g.ext_alg.elt_from_cd({(str(X0),str(X1)):1})
    except(KeyError): pass

    r=g.elt()
    if w0!=None: r+=G.fund_der(G.curvature.apply_cochain_map(w0),i0)
    if w1!=None: r+=G.fund_der(G.curvature.apply_cochain_map(w1),i1)
    if w2!=None: r+=G.fund_der(G.curvature.apply_cochain_map(w2),i2)
    return r

def cb_term(G,i0,i1,i2):
    """returns the coboundary of G.curvature applied to i0,i1,i2 elements of G.symbol.basis.
    This is needed because we care about the value as a cochain in C(g,g), not just C(m,g)
    (at least for the purpose of checks)"""

    r=G.symbol.elt()
    X0,X1,X2=[G.symbol.basis[a] for a in [i0,i1,i2]]
    
    for i in range(3):
        Y0,Y1,Y2=[X0,X1,X2,X0,X1][i:i+3]
        w0=Y0.negative_projection().cast_as_ext_elt().wedge(Y1.negative_projection().cast_as_ext_elt())
        r+=G.curvature.apply_cochain_map(w0).ad(Y2)
        w1=Y0.ad(Y1).negative_projection().cast_as_ext_elt().wedge(Y2.negative_projection().cast_as_ext_elt())
        r+=G.curvature.apply_cochain_map(w1)
    return -r

# What if instead of computing these individually, I just computed Gerstenhaber square
# of curvature once, then 
def gerst_term(G,i0,i1,i2):
    X0,X1,X2=[G.symbol.basis[a] for a in [i0,i1,i2]]
    r=G.symbol.elt()
    for i in range(3):
        Y0,Y1,Y2=[X0,X1,X2,X0,X1][i:i+3]
        Y0=Y0.negative_projection().cast_as_ext_elt()
        Y1=Y1.negative_projection().cast_as_ext_elt()
        Y2=Y2.negative_projection().cast_as_ext_elt()
        w=G.curvature.apply_cochain_map(Y0.wedge(Y1)).negative_projection().cast_as_ext_elt()
        r+=G.curvature.apply_cochain_map(w.wedge(Y2))
    return r

def Bianchi(G,i0,i1,i2,subdivide=False):
    r0=cb_term(G,i0,i1,i2)
    r1=der_term(G,i0,i1,i2)
    r2=gerst_term(G,i0,i1,i2)
    if subdivide: return (r0, r1, r2)
    return r0+r1+r2
    # if subdivide: return (cb_term(G,i0,i1,i2), der_term(G,i0,i1,i2), gerst_term(G,i0,i1,i2))
    # return cb_term(G,i0,i1,i2)+der_term(G,i0,i1,i2)+gerst_term(G,i0,i1,i2)

In [6]:
Bianchi_dict={}

def choose_var(expr,symb,excl_list=[]):
    """Returns a variable of maximal weight from the expression, which should be a rational
    expression in 2-tensors"""
    s=Indexed_obj_in_expr(expr)
    r=next(iter(s))
    for a in s:
        if (wght_of_ind(a,symb)<wght_of_ind(r,symb) and
                r.base[r.indices[0:3]] not in excl_list): 
            r=a
    if r.base[r.indices[0:3]] in excl_list: return None
    return r

def compute_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        time0=time.time()
        print('Computing', t)
        if (i,j,k) in G.Bianchi_cache:
            zero_elt=G.Bianchi_cache[(i,j,k)]
        else: 
            time1=time.time()
            zero_elt=Bianchi(G,i,j,k)
            G.Bianchi_cache[(i,j,k)]=zero_elt
            print('    Bianchi computed in',hrs_min_sec(time.time()-time1))
        time2=time.time()
        to_solve=ds_subs(zero_elt.vec[m],Bianchi_dict,D_G)
        print('    to_solve computed in',hrs_min_sec(time.time()-time2))
        if simplify(to_solve)!=0:
            time3=time.time()
            s=find_a_linear_term(to_solve,G.fund_invars)
            if s==None: s=choose_var(to_solve,g,G.fund_invars)
            if s==None: s=choose_var(to_solve,g)
            sol=solve(to_solve,s,dict=True)[0]
            print('    Solving complete in',hrs_min_sec(time.time()-time3))
            time4=time.time()
            for a in sol: 
                ds_add_key(a,sol[a],Bianchi_dict,D_G)
            print('    Substitution complete in',hrs_min_sec(time.time()-time4))
        print('   ',t,'computed in',hrs_min_sec(time.time()-time0))
        # Notice that D_G.curv = -G.curvature, since I switched sign conventions

def check_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        if (i,j,k) in G.Bianchi_cache:
            zero_elt=G.Bianchi_cache[(i,j,k)]
        else:
            zero_elt=Bianchi(G,i,j,k)
            G.Bianchi_cache[(i,j,k)]=zero_elt
        r=simplify(ds_subs(zero_elt.vec[m],Bianchi_dict,D_G))
        if r!=0: print(r)

In [7]:
for w in range(1,6):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

--------------------- Weight 1 ---------------------
Computing (3, 4, 5, 6)
    Bianchi computed in 3.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 5, 6) computed in 3.0 sec
Computing (3, 4, 6, 7)
    Bianchi computed in 6.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 6, 7) computed in 6.0 sec
Computing (3, 4, 7, 8)
    Bianchi computed in 12.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 7, 8) computed in 12.0 sec
Computing (3, 4, 8, 9)
    Bianchi computed in 15.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 8, 9) computed in 15.0 sec
Computing (3, 4, 9, 10)
    Bianchi computed in 36.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 

In [8]:
eta_to_K={}
for i0 in range(3,len(g.basis)):
    for i1 in range(i0+1,len(g.basis)):
        for i2 in range(len(g.basis)):
            w=g.basis[i0].cast_as_ext_elt().wedge(g.basis[i1].cast_as_ext_elt())
            v=K9.apply_cochain_map(w).vec[i2]
            eta_to_K[IndexedBase('eta')[i0,i1,i2]]=v

def convert_eta_to_K(expr):
    if type(expr)==Indexed:
        if expr.base==IndexedBase('eta'):
            if expr not in eta_to_K:
                prev=convert_eta_to_K(expr.base[expr.indices[0:-1]])
                eta_to_K[expr]=P.fund_der(prev,expr.indices[-1],F=F9)
            return eta_to_K[expr]
        return expr
    s=Indexed_obj_in_expr(expr)
    subs_dict={}
    for A in s:
        subs_dict[A]=convert_eta_to_K(A)
    return expr.xreplace(subs_dict)

In [16]:
for t in [(4, 5, 6, 3)]:
    r=simplify(convert_eta_to_K(G.Bianchi_cache[t[0:3]].vec[t[3]]))
    if r!=0:
        # print(t,'-->',G.Bianchi_cache[t[0:3]].vec[t[3]])
        print(simplify(ds_subs(r,D_JS,D)),'\n')

35*(-3*K[4, 5, 3]*K[4, 8, 6] + K[4, 8, 6, 4, 4] - 11*K[4, 8, 6, 4]*K[5, 8, 9] - 8*K[4, 8, 6, 4]*K[6, 7, 9] + 12*K[4, 8, 6]*K[5, 8, 9]**2 + 13*K[4, 8, 6]*K[5, 8, 9]*K[6, 7, 9] + K[4, 8, 6]*K[6, 7, 9, 4] + 3*K[4, 8, 6]*K[6, 7, 9]**2)*exp(3*e - 11*h)/18 



In [53]:
t=(3,5,6)
X0,X1,X2=[F9.col(a) for a in t]
r=(P.bracket(P.bracket(X0,X1),X2))
r+=(P.bracket(P.bracket(X2,X0),X1))
r+=(P.bracket(P.bracket(X1,X2),X0))
r=simplify(ds_subs(D.normalize_der(r),D_JS,D))
r

Matrix([
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0]])

In [51]:
# This was motivated by the stuff down below; the abstract Bianchi relations seem to be failing in weight 5.
X4,X5,X6=[F9.col(a) for a in [4,5,6]]
r=(P.bracket(P.bracket(X4,X5),X6))
r+=(P.bracket(P.bracket(X6,X4),X5))
r+=(P.bracket(P.bracket(X5,X6),X4))
r=simplify(ds_subs(D.normalize_der(r),D_JS,D))

In [ ]:
r

Matrix([
[(-2116800*y**2*K[4, 5, 3]*K[4, 8, 6] + 705600*y**2*K[4, 8, 6, 4, 4] - 7761600*y**2*K[4, 8, 6, 4]*K[5, 8, 9] - 5644800*y**2*K[4, 8, 6, 4]*K[6, 7, 9] + 8467200*y**2*K[4, 8, 6]*K[5, 8, 9]**2 + 9172800*y**2*K[4, 8, 6]*K[5, 8, 9]*K[6, 7, 9] + 705600*y**2*K[4, 8, 6]*K[6, 7, 9, 4] + 2116800*y**2*K[4, 8, 6]*K[6, 7, 9]**2 + 84672*y*exp(2*h)*K[4, 5, 3]*K[4, 8, 6, 3] - 254016*y*exp(2*h)*K[4, 8, 6, 3, 4, 4] + 2286144*y*exp(2*h)*K[4, 8, 6, 3, 4]*K[5, 8, 9] + 1524096*y*exp(2*h)*K[4, 8, 6, 3, 4]*K[6, 7, 9] + 169344*y*exp(2*h)*K[4, 8, 6, 3]*K[5, 8, 9]**2 + 1072512*y*exp(2*h)*K[4, 8, 6, 3]*K[5, 8, 9]*K[6, 7, 9] - 112896*y*exp(2*h)*K[4, 8, 6, 3]*K[6, 7, 9, 4] + 677376*y*exp(2*h)*K[4, 8, 6, 3]*K[6, 7, 9]**2 - 1552320*y*exp(2*h)*K[4, 8, 6, 4, 3]*K[5, 8, 9] - 1128960*y*exp(2*h)*K[4, 8, 6, 4, 3]*K[6, 7, 9] + 141120*y*exp(2*h)*K[4, 8, 6, 4, 4, 3] - 130536*y*exp(2*h)*K[4, 8, 6, 4]*K[4, 8, 7] + 423360*y*exp(2*h)*K[4, 8, 6, 4]*K[6, 7, 9, 3] - 776160*y*exp(2*h)*K[4, 8, 6, 4]*K[6, 8, 9] - 56448*y*exp(2*

## Recreating Jacobi Failure
I'd like to figure out a simple case of failure similar to the above case

In [180]:
%run Geom_Prolongation.ipynb
g=Symp_symb(7)
E=g.ext_alg
C=g.cochain_complex
D,D_JS=Standard_Prenorm_distr(g,9,unshelve=True)
P=Geom_Prolongation(D,D_JS)
G=RegularCartanGeometry(g,'eta')
D_G=Distr_of_constant_symbol(g,-G.curvature)
y,h,e=symbols('y,h,e')
K=IndexedBase('K')
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis

In [207]:
def wedge(A,B):
    return A.cast_as_ext_elt().wedge(B.cast_as_ext_elt())

simplify(P.Omega_1_curv.apply_cochain_map(wedge(e1,e2)).vec)

Matrix([
[                                                                               -y**2*exp(2*e - 10*h)*K[4, 5, 3]],
[                                                                                   y*exp(2*e - 10*h)*K[4, 5, 3]],
[                                                                                                              0],
[                                                                                     exp(2*e - 10*h)*K[4, 5, 3]],
[(-5*y*(3*K[5, 8, 9] + 2*K[6, 7, 9]) + (5*K[4, 8, 7] + 8*K[6, 7, 9, 3] - 20*K[6, 8, 9])*exp(2*h)/8)*exp(e - 5*h)],
[                                                                     (3*K[5, 8, 9] + 2*K[6, 7, 9])*exp(e - 5*h)],
[                                                                                                              0],
[                                                                                                              0],
[                                                                      

In [193]:
# Nice! This is what I was looking for. Figure out what's going on here.
t=(3,4,5)
X0,X1,X2=[eye(11).col(a) for a in t]
X0=Matrix([[0,0,0,K[4,5,3],0,0,0,0,0,0,0]]).transpose()
r=(P.bracket(P.bracket(X0,X1),X2))
r+=(P.bracket(P.bracket(X2,X0),X1))
r+=(P.bracket(P.bracket(X1,X2),X0))
r=simplify(r)
r

Matrix([
[                                                                                                                                                                                                                   y**2*(K[4, 5, 3, 3] + K[4, 8, 7]*K[5, 8, 9] + K[4, 8, 7]*K[6, 7, 9] + 3*K[5, 8, 9]*K[6, 8, 9] + 2*K[6, 7, 9]*K[6, 8, 9] - K[6, 8, 9, 4])*exp(2*e - 8*h)*K[4, 5, 3]],
[                                                                                                                                                                                                                     y*(-K[4, 5, 3, 3] - K[4, 8, 7]*K[5, 8, 9] - K[4, 8, 7]*K[6, 7, 9] - 3*K[5, 8, 9]*K[6, 8, 9] - 2*K[6, 7, 9]*K[6, 8, 9] + K[6, 8, 9, 4])*exp(2*e - 8*h)*K[4, 5, 3]],
[                                                                                                                                                                                                                                            

In [191]:
P.dir_der(X0,X1)

Matrix([
[                         0],
[                         0],
[                         0],
[exp(e - 5*h)*K[4, 5, 3, 4]],
[                         0],
[                         0],
[                         0],
[                         0],
[                         0],
[                         0],
[                         0]])

In [212]:
(h*H+e*E).Ad((y*Y).Ad(X))

-y**2*exp(-2*h)*Y + -y*H + exp(2*h)*X

In [209]:
P.dir_der(K[4,5,3],X1)

exp(e - 5*h)*K[4, 5, 3, 4]

In [189]:
P.Omega_1_bracket(X0,X1)-P.dir_der(X0,X1)+P.dir_der(X1,X0)

Matrix([
[                          0],
[                          0],
[                          0],
[-exp(e - 5*h)*K[4, 5, 3, 4]],
[                          0],
[                 K[4, 5, 3]],
[                          0],
[                          0],
[                          0],
[                          0],
[                          0]])

In [188]:
P.bracket(X0,X1)

Matrix([
[                          0],
[                          0],
[                          0],
[-exp(e - 5*h)*K[4, 5, 3, 4]],
[                          0],
[                 K[4, 5, 3]],
[                          0],
[                          0],
[                          0],
[                          0],
[                          0]])

In [13]:
for t in tuples_by_wght[5]:
    r=simplify(ds_subs(convert_eta_to_K(G.Bianchi_cache[t[0:3]].vec[t[3]]),D_JS,D))
    if r!=0:
        print(t,'-->',G.Bianchi_cache[t[0:3]].vec[t[3]])
        print(r,'\n')

(4, 5, 6, 3) --> (-2*eta[3, 9, 9] - eta[5, 9, 10]/5)*eta[4, 5, 3, 3, 4] + (2*eta[3, 9, 9] + eta[5, 9, 10]/5)*eta[4, 5, 3, 4, 3] + ((eta[3, 9, 9] + eta[5, 9, 10]/5)*eta[3, 9, 9] + eta[3, 9, 9, 3] - eta[6, 9, 10]/8 + 5*eta[7, 8, 10]/72)*eta[4, 5, 3, 4] - (-5*eta[4, 6, 5]/8 - eta[5, 6, 6] + eta[6, 7, 8] + 5*eta[6, 8, 9]/8)*eta[4, 6, 3] + (-5*eta[4, 7, 5]/9 - 8*eta[5, 7, 6]/9 - eta[6, 7, 7] + 5*eta[7, 8, 9]/9)*eta[4, 5, 3] + (-eta[4, 6, 6] + eta[4, 8, 8] + eta[4, 9, 9] - 9*eta[5, 6, 7]/5 - 8*eta[5, 7, 8]/5 - eta[6, 7, 9])*eta[5, 6, 3] - (eta[4, 6, 6] - eta[4, 8, 8] - eta[4, 9, 9] + 18*eta[5, 6, 7]/5 + 16*eta[5, 7, 8]/5 + eta[5, 8, 9] + eta[6, 7, 9])*eta[4, 6, 3, 3] + (eta[4, 6, 6] - eta[4, 8, 8] - eta[4, 9, 9] + 18*eta[5, 6, 7]/5 + 16*eta[5, 7, 8]/5 + eta[5, 8, 9] + eta[6, 7, 9])*eta[5, 6, 3] - 8*(eta[4, 6, 4]/8 + eta[4, 7, 5]/36 + 11*eta[5, 6, 5]/40 + 11*eta[5, 7, 6]/45 + eta[5, 8, 7]/5 + eta[5, 9, 8]/5 + 3*eta[6, 7, 7]/40 + 3*eta[6, 8, 8]/40 + eta[6, 9, 9]/8 - eta[7, 8, 9]/36)*eta[4, 5, 

## Checking Bianchi for Normal Frame Directly

In [10]:
P.Bianchi(F9,K9,3,4,5,6)

Y0 = X
Y1 = (-y*(K[5, 8, 9] + K[6, 7, 9])*exp(e - 5*h) - exp(e - 3*h)*K[4, 8, 7]/8)*Y + (K[5, 8, 9] + K[6, 7, 9])*exp(e - 5*h)/2*H + (-5*K[5, 8, 9] - 3*K[6, 7, 9])*exp(e - 5*h)/2*E + e_1
Y2 = (4*y**2*(-K[5, 8, 9] - K[6, 7, 9])*exp(e - 5*h) + y*(-13*K[4, 8, 7] - 13*K[6, 8, 9])*exp(e - 3*h)/26 - exp(e - h)*K[4, 8, 7, 3]/8)*Y + (3*y*(K[5, 8, 9] + K[6, 7, 9])*exp(e - 5*h)/2 + (-26*K[4, 8, 7] + 104*K[6, 8, 9])*exp(e - 3*h)/416)*H + (-5*y*(5*K[5, 8, 9] + 3*K[6, 7, 9])*exp(e - 5*h)/2 + (-130*K[4, 8, 7] + 416*K[6, 7, 9, 3] - 520*K[6, 8, 9])*exp(e - 3*h)/416)*E + (-K[5, 8, 9] - K[6, 7, 9])*exp(e - 5*h)*X + e_2
K.apply_cochain_map(Y0.wedge(Y1)) = 0
w = 0
Y0 = (-y*(K[5, 8, 9] + K[6, 7, 9])*exp(e - 5*h) - exp(e - 3*h)*K[4, 8, 7]/8)*Y + (K[5, 8, 9] + K[6, 7, 9])*exp(e - 5*h)/2*H + (-5*K[5, 8, 9] - 3*K[6, 7, 9])*exp(e - 5*h)/2*E + e_1
Y1 = (4*y**2*(-K[5, 8, 9] - K[6, 7, 9])*exp(e - 5*h) + y*(-13*K[4, 8, 7] - 13*K[6, 8, 9])*exp(e - 3*h)/26 - exp(e - h)*K[4, 8, 7, 3]/8)*Y + (3*y*(K[5, 8, 9] + K[6, 7, 

0

In [ ]:
for t in tuples_by_wght[1]:
    print(t,'-->',simplify(P.Bianchi(F9,K9,*t)))

(3, 4, 5, 6)


invalid_parent_exception: wedge recieved arguments with incompatible parents

## Checking Jacobi/Bianchi agreement for D

In [265]:
Jacobi_by_wght={}
for k in D_JS:
    for j in D_JS[k]:
        w=-wght_of_ind(k.base[k.indices+j],g)
        if not w in Jacobi_by_wght: Jacobi_by_wght[w]=[]
        new_syzygy=simplify(k.base[k.indices+j]-D_JS[k][j])
        if new_syzygy!=0:
            new_syzygy=new_syzygy*new_syzygy.as_numer_denom()[1]
            Jacobi_by_wght[w].append(new_syzygy)

In [268]:
temp_D_cache={}
temp_Bianchi_cache={}
for i0 in range(3,len(g.basis)):
    for i1 in range(i0+1,len(g.basis)):
        for i2 in range(i1+1,len(g.basis)):
            for i3 in range(len(g.basis)):
                    w0,w1,w2,w3=[g.basis[a].wght for a in [i0,i1,i2,i3]]
                    if w3-w0-w1-w2<=5:
                        temp_D_cache[(i0,i1,i2,i3)]=simplify(ds_subs((D.Bianchi(i0,i1,i2,i3)),D_JS,D))

In [269]:
# Do Jacobi and Bianchi from D agree?
for t in temp_D_cache:
    ji=ds_subs(D.Jacobi_id(*t),D_JS,D)
    if simplify(temp_D_cache[t]-ji)!=0:
        print('t =',t)
        print('Bianchi:',temp_D_cache[t])
        print('Jacobi:',ji,'\n')

In [270]:
for t in temp_D_cache:
    if temp_D_cache[t]!=0:
        w0,w1,w2,w3=[g.basis[a].wght for a in t]
        print(t,'of wght',w3-w0-w1-w2,'-->',temp_D_cache[t],'\n')

In [271]:
temp_Bianchi_cache={}
for i0 in range(len(g.basis)):
    for i1 in range(i0+1,len(g.basis)):
        for i2 in range(i1+1,len(g.basis)):
            for i3 in range(len(g.basis)):
                    w0,w1,w2,w3=[g.basis[a].wght for a in [i0,i1,i2,i3]]
                    if w3-w0-w1-w2<=5:
                        temp_Bianchi_cache[(i0,i1,i2,i3)]=simplify(ds_subs((P.Bianchi(eye(len(g.basis)),P.Omega_1_curv,i0,i1,i2,i3)),D_JS,D))

In [272]:
for t in temp_Bianchi_cache:
    s=simplify(temp_Bianchi_cache[t])
    if s!=0:
        w0,w1,w2,w3=[g.basis[a].wght for a in t]
        print(t,'of wght',w3-w0-w1-w2,'-->',s,'\n')